# Figure 3 — Convergence Rates

Reproduces **Figure 3** from *Density-Reweighted Entropic Optimal Transport*.

**Experiment:** Vary m ∈ [100, 10000] (log-uniform, 10 values), n = 2m, 20 independent replicates per m. Compute the DR-EOT plan W^(θ=1) and compare entry-wise to the population-level plan W_ε computed on a fine uniform grid (N=10000 arc-length-uniform points).

**Error metrics:**
- L1: (1/mn) Σ |√(Vol_X · Vol_Y) · W_ij − W_ε(x_i, y_j)|
- L∞: max_ij |√(Vol_X · Vol_Y) · W_ij − W_ε(x_i, y_j)|

Both metrics decay at rate m^{-1/2}.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from sklearn.metrics import pairwise_distances
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import make_interp_spline
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from dreot import sinkhorn_eot, sinkhorn_dreot

## 1. Samplers (same as Figures 1–2)

In [ ]:
class PiecewiseUniform:
    def __init__(self, break_point=0.5, weights=(1, 1)):
        self.bp = break_point
        prob = break_point * weights[0] + (1 - break_point) * weights[1]
        self.w = [weights[0] / prob, weights[1] / prob]
        self.coin = break_point * weights[0] / prob

    def sample(self, n):
        c = np.random.uniform(size=n)
        n1 = int(np.sum(c <= self.coin))
        x = np.concatenate([
            np.random.uniform(0, self.bp, n1),
            np.random.uniform(self.bp, 1, n - n1),
        ])
        return np.sort(x).reshape(-1, 1)

    def pdf(self, x):
        return np.where(np.asarray(x) < self.bp, self.w[0], self.w[1])


samplerX = PiecewiseUniform(0.5, (9, 1))
samplerY = PiecewiseUniform(0.5, (1, 9))
c_curve  = 0.5
EPS      = 5e-2
K_RATIO  = 2    # n = 2m
N_TRIALS = 20
m_vals   = np.unique(np.logspace(np.log10(100), np.log10(10000), 10).astype(int))

# Manifold arc-lengths (analytic)
VOL_X = np.sqrt(2)   # line segment: |γ'(t)| = sqrt(2) on [0,1]
t_fine = np.linspace(0.0, 1.0, 200_000)
arc_ds = np.sqrt(1 + (1 + 2*c_curve*t_fine)**2)
VOL_Y  = np.trapz(arc_ds, t_fine)

print(f"m values : {m_vals}")
print(f"Vol_X = {VOL_X:.4f}  Vol_Y = {VOL_Y:.4f}")

## 2. Population-level plan via fine grid + cubic spline interpolation

In [ ]:
N_GRID = 10_000

# X manifold: arc-length uniform ↔ equally spaced t (ds/dt = sqrt(2) = const)
t_unif_X = np.linspace(0.0, 1.0, N_GRID)
X_unif   = np.column_stack([t_unif_X, t_unif_X])

# Y manifold: invert arc-length CDF to get arc-length-uniform t
arc_cdf = np.concatenate([[0.0], cumulative_trapezoid(arc_ds, t_fine)])
s_tgts  = np.linspace(0.0, arc_cdf[-1], N_GRID)
t_unif_Y = np.interp(s_tgts, arc_cdf, t_fine)
Y_unif   = np.column_stack([t_unif_Y, 2 + t_unif_Y + c_curve * t_unif_Y**2])

# Standard Sinkhorn on fine grid (uniform marginals → population plan)
D_unif   = pairwise_distances(X_unif, Y_unif, metric="sqeuclidean")
r_unif   = np.ones((N_GRID, 1)) * N_GRID
c_unif   = np.ones((N_GRID, 1)) * N_GRID

print("Running Sinkhorn on fine grid (this may take ~1 min) ...")
u_hat, v_hat = sinkhorn_eot(
    D_unif, EPS, r_unif, c_unif,
    delta=1e-6, max_iter=5000, check_freq=100, raise_on_bad_convergence=False,
)

# Cubic-spline extensions of the scaling vectors
interp_u = make_interp_spline(t_unif_X, u_hat.ravel(), k=3)
interp_v = make_interp_spline(t_unif_Y, v_hat.ravel(), k=3)
print("Population plan computed.")

## 3. Convergence experiment loop

In [ ]:
np.random.seed(42)
scale = np.sqrt(VOL_X * VOL_Y)

err_L1   = np.zeros((len(m_vals), N_TRIALS))
err_Linf = np.zeros((len(m_vals), N_TRIALS))

for j, m in enumerate(m_vals):
    n = K_RATIO * m
    for rep in tqdm(range(N_TRIALS), desc=f"m={m}"):
        xA = samplerX.sample(m).ravel()
        xB = samplerY.sample(n).ravel()

        XA = np.column_stack([xA, xA])
        XB = np.column_stack([xB, 2 + xB + c_curve * xB**2])

        D = pairwise_distances(XA, XB, metric="sqeuclidean")

        mu_arc = samplerX.pdf(xA) / np.sqrt(2)
        nu_arc = samplerY.pdf(xB) / np.sqrt(1 + (1 + 2*c_curve*xB)**2)

        row_s, col_s = sinkhorn_dreot(
            D, EPS,
            mu_arc.reshape(-1, 1), nu_arc.reshape(-1, 1),
            alpha=1,
            delta=1e-6, max_iter=1000, check_freq=100,
            raise_on_bad_convergence=False,
        )
        W_emp = row_s * np.exp(-D / EPS) * col_s.T

        # Population plan evaluated at sample points
        W_pop = (interp_u(xA).reshape(-1, 1)
                 * np.exp(-D / EPS)
                 * interp_v(xB).reshape(1, -1))

        diff = np.abs(scale * W_emp - W_pop)
        err_L1[j, rep]   = diff.mean()
        err_Linf[j, rep] = diff.max()

mean_L1   = err_L1.mean(axis=1)
mean_Linf = err_Linf.mean(axis=1)
print("Done.")

## 4. Figure 3 — log-log convergence plot

In [ ]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "axes.labelsize": 10, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "legend.fontsize": 9, "axes.grid": True, "grid.alpha": 0.3,
    "lines.linewidth": 2, "figure.dpi": 150, "savefig.dpi": 600,
    "pdf.fonttype": 42,
})

# Fitted slopes
log_m = np.log(m_vals)
slope_L1,   _ = np.polyfit(log_m, np.log(mean_L1),   1)
slope_Linf, _ = np.polyfit(log_m, np.log(mean_Linf), 1)

# Reference m^{-1/2} lines anchored at the last data point
ref_L1   = mean_L1[-1]   * np.sqrt(m_vals[-1] / m_vals)
ref_Linf = mean_Linf[-1] * np.sqrt(m_vals[-1] / m_vals)

fig, ax = plt.subplots(1, 1, figsize=(5, 4))

ax.loglog(m_vals, mean_L1,   "o-",
          label=rf"$L_1$ error  (slope≈{slope_L1:.2f})")
ax.loglog(m_vals, mean_Linf, "s-",
          label=rf"$L_\infty$ error  (slope≈{slope_Linf:.2f})")
ax.loglog(m_vals, ref_L1,   "--", label=r"$\propto m^{-1/2}$")
ax.loglog(m_vals, ref_Linf, "--")

ax.set_xlabel(r"$m$")
ax.set_ylabel("Approximation Error")
ax.grid(which="minor", linestyle=":", linewidth=0.5, alpha=0.5)
ax.minorticks_on()
ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("fig3_convergence.pdf", bbox_inches="tight", dpi=600)
plt.show()
print("Saved fig3_convergence.pdf")